# 🧠 E-Ticaret Metin Yazarı Fine-Tuning (Unsloth QLoRA)
Bu notebook, Llama 3.1 8B modelini `train.jsonl` verisi ile eğitip GGUF olarak dışa aktarmak içindir.

**Google Colab üzerinde T4 GPU ile çalıştırınız.**

### Çalıştırmadan önce:
1. Üst menüden `Çalışma Zamanı` → `Çalışma zamanı türünü değiştir` → **T4 GPU** seçin
2. Sol paneldeki Klasör ikonuna tıklayıp `train.jsonl` dosyasını yükleyin
3. `Çalışma Zamanı` → **Tümünü Çalıştır** deyin

## 💾 Faz 0: Google Drive Bağlantısı (Zorunlu - Dosyaların Kaybolmaması İçin)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive bağlantısı başarılı!")

## 🔧 Faz 1: Ortam Kurulumu

In [ ]:
# OOM (Out of Memory) hatalarını önlemek için PyTorch bellek yapılandırması
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# T4 GPU (15GB VRAM) ile güvenli çalışmak için 512 token yeterli
# E-ticaret ürün açıklamaları ortalama 150-300 token, max 500 token
max_seq_length = 512
dtype = None       # Otomatik algılama (T4 icin fp16)
load_in_4bit = True  # 4-bit kuantizasyon ile 6GB VRAM kullanımı

print(f"CUDA mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 🧠 Faz 2: Modelin Yüklenmesi ve LoRA Konfigürasyonu

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model yüklendi!")

In [ ]:
# LoRA adaptörlerini modele ekle
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # LoRA rank - kapasite
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,           # Öğrenme şiddeti
    lora_dropout = 0,          # 0 = Unsloth optimize ediyor
    bias = "none",             # 0 = Unsloth optimize ediyor
    use_gradient_checkpointing = "unsloth",  # VRAM kullanımını %30 azaltır
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("LoRA adaptörleri eklendi!")

## 📚 Faz 3: Veri Hazırlığı ve Filtreleme
**ÖNEMLİ:** Sol taraftaki Klasör ikonuna tıklayıp `train.jsonl` dosyasını Colab'e sürükleyip bırakın.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Llama 3.1 ChatML formatını tokenizer'a yükle
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Veri setini yükle ve ChatML formatına dönüştür
dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True, desc="ChatML formatına dönüştürme")

# ÇOK ÖNEMLİ: max_seq_length'i aşan örnekleri filtrele
# Bu olmadan token uyumsuzluğu (ValueError) hatası alınır!
def filter_long_examples(example):
    tokens = tokenizer(example["text"], return_tensors=None)
    return len(tokens["input_ids"]) <= max_seq_length

before = len(dataset)
dataset = dataset.filter(filter_long_examples, desc="Uzun örnekleri filtreleme")
after = len(dataset)

print(f"Filtreleme tamamlandı: {before} -> {after} örnek ({before - after} örnek çıkarıldı)")
print(f"\nÖrnek çıktı kontrol\u00fc:\n{dataset[0]['text'][:400]}")

## 🚂 Faz 4: SFTTrainer Eğitimi (~35-40 dakika)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,   # T4 GPU OOM önlemek için 1
        gradient_accumulation_steps = 8,   # Efektif batch size = 1x8 = 8
        num_train_epochs = 1,              # Tam 1 tur eğitim
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",              # Checkpoint kaydetme (PicklingError önlemi)
    ),
)

trainer_stats = trainer.train()
print(f"Eğitim tamamlandı! Süre: {trainer_stats.metrics['train_runtime']/60:.1f} dakika")

## 📦 Faz 5: Google Drive'a Kaydet ve GGUF Export
Bu hücre, eğitilmiş modeli Google Drive'a kaydeder. Colab kapansa bile dosyalar kaybolmaz!

In [ ]:
# LoRA adaptörlerini Drive'a kaydet (hızlı yedek - ~200MB)
DRIVE_PATH = "/content/drive/MyDrive/eticaret-llm"
os.makedirs(DRIVE_PATH, exist_ok=True)

print("LoRA adaptörleri kaydediliyor...")
model.save_pretrained(f"{DRIVE_PATH}/lora_model")
tokenizer.save_pretrained(f"{DRIVE_PATH}/lora_model")
print("LoRA kaydedildi!")

# Modeli GGUF formatına çevirip Drive'a kaydet (~10-15 dakika)
print("\nGGUF formatına dönüştürülüyor (10-15 dakika sürebilir)...")
model.save_pretrained_gguf(
    f"{DRIVE_PATH}/model_gguf",
    tokenizer,
    quantization_method = "q4_k_m"
)
print(f"\nBITTI! Dosyalar Google Drive'da:\n{DRIVE_PATH}/model_gguf/")